In [1]:
#pip install scvelo

In [2]:
#pip install python-igraph

In [3]:
#pip install louvain

In [4]:
#pip install scanpy[leiden]

In [5]:
import pandas as pd
import scanpy as sc
import numpy as np
from numpy import linalg as la
from scipy.sparse import issparse
import loompy
import anndata
import scvelo as scv
scv.logging.print_version()

Running scvelo 0.3.4 (Python 3.10.21) on 2026-09-30 14:41.


In [6]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

KeyboardInterrupt: 

In [ ]:
import os
os.chdir('/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo/')

In [ ]:
## integrated h5ad in the dynamo script 
adata =scv.read("normal-CIS_integrated.h5ad")

In [ ]:
adata

In [ ]:
adata.var_names_make_unique()
adata # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`

In [ ]:
scv.pp.filter_genes(adata, min_shared_counts=20)
scv.pp.normalize_per_cell(adata)
scv.pp.filter_genes_dispersion(adata, n_top_genes=2000)
scv.pp.log1p(adata)
adata

In [ ]:
#scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
#scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
##n_neighbors=7 test ###This was the best
## n_pcs increased to make more clusters =50
#scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
#scv.pp.moments(adata, n_pcs=50, n_neighbors=5)
#scv.pp.log1p(adata)

In [ ]:
##n_neighbors=10 test ###This was the best
## n_pcs increased to make more clusters =50

scv.pp.moments(adata, n_pcs=50, n_neighbors=10)
scv.pp.log1p(adata)

In [ ]:
scv.pp.neighbors(adata, n_neighbors=10, n_pcs=None, use_rep=None, use_highly_variable=True, knn=True, random_state=0, method='umap', metric='euclidean', metric_kwds=None, num_threads=-1, copy=False)

In [ ]:
#scv.pp.remove_duplicate_cells(adata) 

In [ ]:
##n_neighbors=100 test
#scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
#scv.pp.moments(adata, n_pcs=30, n_neighbors=100)

In [ ]:
adata

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=40)
scv.tl.velocity(adata, mode='dynamical')
scv.tl.velocity_graph(adata, n_jobs=40)

In [ ]:
#scv.pl.velocity_embedding_stream(adata, basis='pca')

In [ ]:
scv.tl.umap(adata)

In [ ]:
adata

In [ ]:
#adata.obsm['X_umap'] = adata.obsm['umap_cell_embeddings']

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='X_umap') 

In [ ]:
scv.pl.velocity(adata, ['Mki67', 'Trp63', 'Krt13'], ncols=2, basis='X_umap')

In [ ]:
adata.write('normal-CIS_n10pca50.h5ad', compression='gzip')

In [ ]:
adata_1 = sc.read('normal-CIS_n10pca50.h5ad')    ## begin with .h5ad, use scanpy codes for clustering

In [ ]:
sc.settings.verbosity = 1             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor='white')

In [ ]:
adata_1

In [ ]:
#pip install python-igraph
#pip install scanpy[leiden]
#pip install louvain

In [ ]:
sc.tl.louvain(adata_1)

In [ ]:
sc.pl.umap(adata_1, color='louvain')

In [ ]:
adata_1

In [ ]:
sc.tl.rank_genes_groups(adata_1, 'louvain', method='t-test')
sc.pl.rank_genes_groups(adata_1, n_genes=25, sharey=False)

In [ ]:
sc.settings.verbosity = 2  # reduce the verbosity

In [ ]:
sc.tl.rank_genes_groups(adata_1, 'louvain', method='wilcoxon')
sc.pl.rank_genes_groups(adata_1, n_genes=25, sharey=False)

In [ ]:
df=pd.DataFrame(adata_1.uns['rank_genes_groups']['names'])
df.to_csv("/Users/kko1/Documents/Experiments/Results/scRNAseq/Nat.Comm[2020]._public/Dynamo/markergene.csv",
                 sep=',', na_rep='', float_format=None, columns=None, header=True, 
                 index=True, index_label=None, mode='w', encoding=None, compression='infer', 
                 quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None)

In [ ]:
sc.pl.umap(adata_1, color=['louvain','batch'], legend_loc='right margin', title='',frameon=False, save='.pdf')

In [ ]:
sc.pl.umap(adata_1, color=['louvain'], legend_loc='on data', title='',frameon=False)

In [ ]:
adata_1.write('normal-CIS_before_annotation.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata_1, color='louvain', legend_loc='on data', title='',frameon=False, save='.pdf')

In [ ]:
new_cluster_names = ['Stratified_1','Proliferating_1',
                     'New_1','New_2','New_3',
                     'Stratified_2','Proliferating_2',
                     'New_4','New_5',
                     'Proliferating_3','Stratified_3',
                     'New_6','Fibro_1','New_7','Fibro_2',
                     'Follicular','New_8','New_9',
                     'Fibro_3','Fibro_4','Stratified_4','New_10','New_11','Proliferating_4'
                      ]
adata_1.rename_categories('louvain', new_cluster_names)

In [ ]:
adata_1.obs['louvain']

In [ ]:
adata_1.obs['louvain'].cat.reorder_categories(['Proliferating_1','Proliferating_2','Proliferating_3','Proliferating_4',
                                               'Stratified_1','Stratified_2','Stratified_3','Stratified_4',
                                               'New_1','New_2','New_3','New_4','New_5','New_6',
                                               'New_7','New_8','New_9','New_10','New_11',
                                               'Fibro_1','Fibro_2','Fibro_3','Fibro_4','Follicular'
                                              ],inplace=True)
sc.pl.umap(adata_1, color='louvain', legend_loc='right margin')

In [ ]:
adata_1.uns['louvain_colors']

In [ ]:
adata_1.uns['louvain_colors']= ['#31c3e8','#9aa4fc','#667fff','#4871b8',
                                '#fa4d4d','#F10934','#F78310','#eb4b6e',
                               '#edca58','#edb158','#cced55','#7bbd46','#26961a','#52ab5c',
                                '#4fa889','#206154','#9bc98f','#adc98f','#f2f2a2',
                                '#fc03ba','#f5b5f2','#f235e9','#bf309e','#ad84a4']
                                                                                 
adata_1.uns['louvain_colors']

In [ ]:
sc.pl.umap(adata_1, color='louvain', size=10, legend_loc='right margin', title='umap_Inflammation',frameon=False, save='umap_annotated.pdf')

In [ ]:
sc.pl.umap(adata_1, color=['louvain'], size=10, title='umap_Integrated',frameon=False,legend_loc='on data',legend_fontsize=9, save='umap_annotated_label.pdf')

In [ ]:
sc.pl.umap(adata_1, color=['batch'], size=10, title='umap_Integrated',frameon=False,legend_loc='right margin',legend_fontsize=9, save='umap_annotated_batch_label.pdf')

In [ ]:
new_cluster_names = ['Epi_1','Epi_2','Epi_3','Epi_4','Epi_5','Epi_6','Epi_7','Epi_8',
                    'New_1','New_2','New_3','New_4','New_5','New_6',
                     'New_7','New_8','New_9','New_10','New_11',
                     'Fibro_1','Fibro_2','Fibro_3','Fibro_4','Other'
                     ]
adata_1.rename_categories('louvain', new_cluster_names)

In [ ]:
sc.pl.umap(adata_1, color='louvain', size=10, legend_loc='right margin', title='umap_integrated',frameon=False, save='umap_annotated_1.pdf')

In [ ]:
adata_1

In [ ]:
scv.tl.velocity_graph(adata_1)

In [ ]:
scv.pl.scatter(adata_1, groups=[[c] for c in adata_1.obs['batch'].cat.categories], color='batch', ncols=4, save="scatterplot_batch")

In [ ]:
### plots before applying "scv.tl.recover_dynamics(adata_1)"
#scv.pl.velocity_embedding_stream(adata_1, basis='pca', legend_loc='right margin', frameon=False)

In [ ]:
scv.pl.velocity_embedding_stream(adata_1, basis='umap', legend_loc='right margin')

In [ ]:
scv.pl.velocity_embedding(adata_1,arrow_length=5, arrow_size=2, dpi=120, legend_loc="right margin")

In [ ]:
#scv.pl.velocity_embedding(adata_1,arrow_length=5, arrow_size=2, basis='pca',dpi=120, legend_loc="right margin")

In [ ]:
adata_1.write('Integrated_annotated.h5ad', compression='gzip')

In [ ]:
#adata_2 = sc.read('PC_combined_2nd_1.h5ad')

In [ ]:
####### applying "scv.tl.recover_dynamics(adata_1)" #######
#scv.tl.recover_dynamics(adata_1)             
#scv.tl.velocity(adata_1, mode='dynamical')

In [ ]:
#### random state (in the terminal_state) tested - no effect####
### n_neighbors (in the velocity_graph) test - we can optimize the n_neighbors #####

scv.tl.velocity_graph(adata_1, n_neighbors=10, mode_neighbors='connectivities')
scv.tl.terminal_states(adata_1)
scv.tl.latent_time(adata_1)

scv.pl.scatter(adata_1, color='velocity_pseudotime', cmap='gnuplot',save='velocity_pseudotime.pdf')
scv.pl.scatter(adata_1, color=['root_cells', 'end_points'], save='root_endpoint.pdf')
scv.pl.scatter(adata_1, color='latent_time', cmap='gnuplot', save='latent_time.pdf')



In [ ]:
#### random state (in the terminal_state) tested - no effect####
### n_neighbors (in the velocity_graph) test - we can optimize the n_neighbors #####

scv.tl.velocity_graph(adata_1,n_neighbors=5, mode_neighbors='connectivities')
scv.tl.terminal_states(adata_1)
scv.tl.latent_time(adata_1)

scv.pl.scatter(adata_1, color='velocity_pseudotime', cmap='gnuplot')
scv.pl.scatter(adata_1, color=['root_cells', 'end_points'])
scv.pl.scatter(adata_1, color='latent_time', cmap='gnuplot')

In [ ]:
#scv.pl.velocity_embedding(adata_1,arrow_length=5, arrow_size=4, dpi=120, legend_loc="right margin", save="_arrow_umap.pdf")

In [ ]:
scv.pl.velocity_embedding(adata_1,arrow_length=10, arrow_size=2, dpi=120, legend_loc="right margin",save="_arrow_umap_1.pdf")

In [ ]:
scv.pl.velocity_embedding_stream(adata_1, basis='umap', legend_loc='right margin', save="_stream_umap.svg")

In [ ]:
# n =10 
scv.pl.velocity_embedding_stream(adata_1, density=2, min_mass=0 
                                 ,cutoff_perc=0,arrow_size=0.75, 
                                 basis='umap',frameon=False ,legend_loc='right margin', 
                                 size=20, save='_stream_umap_1.svg')

In [ ]:
# n= 5
scv.pl.velocity_embedding_stream(adata_1, density=2, min_mass=0 
                                 ,cutoff_perc=0,arrow_size=0.75, 
                                 basis='umap',frameon=False ,legend_loc='right margin', 
                                 size=20)

In [ ]:
scv.pl.velocity_embedding_stream(adata_1, density=2, min_mass=1,cutoff_perc=0 , legend_loc='right margin', arrow_size=0.75, size=5 )

In [ ]:
scv.pl.velocity_embedding_stream(adata_1, density=2, min_mass=1,cutoff_perc=0 , legend_loc='on data',legend_fontsize=7, arrow_size=0.75, size=5 )

In [ ]:
#scv.pl.velocity_embedding(adata_1,basis='pca', arrow_length=10, arrow_size=2, dpi=120, legend_loc="right margin", save="_arrow_pca.pdf")

In [ ]:
scv.pl.velocity_embedding_stream(adata_1, basis='pca',density=2, min_mass=0 
                                 ,cutoff_perc=0,arrow_size=0.75,size=20, legend_loc='on data',legend_fontsize=7, save="_stream_pca.pdf")

In [ ]:
print(adata_1.var['velocity_genes'].sum(), adata_1.n_vars)

In [ ]:
top_genes = adata_1.var.sort_values('fit_likelihood', ascending=False).index[:10]

In [ ]:
#Hyperplasia
#scv.pl.velocity(adata_1, ['Mki63','Trp63','Sox2','Top2a','Krt15', 'Krt13','Krt5'], ncols=2)

In [ ]:
scv.pl.velocity(adata_1, ['Mki63','Trp63','Sox2','Top2a','Krt15', 'Krt13','Krt5'], ncols=2)

In [ ]:
scv.tl.score_genes_cell_cycle(adata_1)
scv.pl.scatter(adata_1, color_gradients=['S_score', 'G2M_score'], smooth=True, perc=[5, 95], save="cellcycle.pdf")

In [ ]:
scv.tl.velocity_confidence(adata_1)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata_1, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
df = adata_1.obs.groupby('louvain')[keys].mean().T
df.style.background_gradient(cmap='coolwarm', axis=1)

In [ ]:
scv.pl.velocity_graph(adata_1, threshold=.1, legend_loc="right margin")

In [ ]:
x, y = scv.utils.get_cell_transitions(adata_1, basis='umap', starting_cell=70)
ax = scv.pl.velocity_graph(adata_1, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata_1, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

In [ ]:
# PAGA requires to install igraph, if not done yet.
##!pip install python-igraph --upgrade --quiet

In [ ]:
# this is needed due to a current bug - bugfix is coming soon.
adata_1.uns['neighbors']['distances'] = adata_1.obsp['distances']
adata_1.uns['neighbors']['connectivities'] = adata_1.obsp['connectivities']

scv.tl.paga(adata_1, groups='louvain')
df = scv.get_df(adata_1, 'paga/transitions_confidence', precision=2).T
df.style.background_gradient(cmap='Blues').format('{:.2g}')


In [ ]:
scv.pl.paga(adata_1, basis='umap', size=10, alpha=.1, color='louvain',
            min_edge_width=1, node_size_scale=1.5, save='CIS_paga.pdf')

In [ ]:
df = adata_1.var
df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

kwargs = dict(xscale='log', fontsize=16)
with scv.GridSpec(ncols=3) as pl:
    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

scv.get_df(adata_1, 'fit*', dropna=True).head()

In [ ]:
scv.pl.scatter(adata_1, color='latent_time', color_map='gnuplot', size=80)

In [ ]:
top_genes = adata_1.var['fit_likelihood'].sort_values(ascending=False).index[:300]
scv.pl.heatmap(adata_1, var_names=top_genes, sortby='latent_time', col_color='louvain', n_convolve=100, save=".pdf")

In [ ]:
adata_1.write('Integrated_final.h5ad', compression='gzip')

In [ ]:
#adata_1 = sc.read('PC_merge_2ndtry_final.h5ad') 

In [ ]:
!pip install cellrank --upgrade --quiet
import cellrank as cr

In [ ]:
cr.tl.initial_states(adata_1, cluster_key='louvain',n_cells=30, softmax_scale=4, n_state=1)
cr.pl.initial_states(adata_1, discrete=True)

In [ ]:
cr.tl.terminal_states(adata_1, cluster_key='louvain',n_cells=5, softmax_scale=4, n_state=1)
cr.pl.terminal_states(adata_1, discrete=True)

In [ ]:
cr.tl.lineages(adata_1)
cr.pl.lineages(adata_1)

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/theislab/cellrank@dev

In [ ]:
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np

scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo")
cr.settings.verbosity = 2

In [ ]:
adata_1

In [ ]:
scv.pl.scatter(adata_1, basis="umap", c="louvain", legend_loc="right")

In [ ]:
# filter, normalize total counts and log-transform
sc.pp.filter_genes(adata_1, min_cells=10)
scv.pp.normalize_per_cell(adata_1)
sc.pp.log1p(adata_1)

# hvg annotation
sc.pp.highly_variable_genes(adata_1)
print(f"This detected {np.sum(adata_1.var['highly_variable'])} highly variable genes. ")

# use scVelo's `moments` function for imputation - note that hack we're using here:
# we're copying our `.X` matrix into the layers because that's where `scv.tl.moments`
# expects to find counts for imputation
adata_1.layers["spliced"] = adata_1.X
adata_1.layers["unspliced"] = adata_1.X
scv.pp.moments(adata_1, n_pcs=30, n_neighbors=30)

In [ ]:
# import and initialize the CytoTRACE kernel, compute transition matrix - done!
import scvelo as scv
# CytoTRACE by default uses imputed data - a simple way to compute KNN-imputed data is to use scVelo's moments
# function. However, note that this function expects `spliced` counts because it's designed for RNA velocity,
# so we're using a simple hack here:
if 'spliced' not in adata_1.layers or 'unspliced' not in adata_1.layers:
    adata_1.layers['spliced'] = adata_1.X
    adata_1.layers['unspliced'] = adata_1.X

# compute KNN-imputation using scVelo's moments function
scv.pp.moments(adata)
import cellrank as cr
from cellrank.tl.kernels import CytoTRACEKernel
ctk = CytoTRACEKernel(adata_1)